In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import json
import time
from datetime import datetime

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
# 문맥과 대화에 대한 내용을 이해하는 것
messages = []

messages.append()

In [2]:
llm.invoke('내 이름은 abc 입니다.').content

'안녕하세요, abc님! 어떻게 도와드릴까요?'

In [3]:
# 위에서 물어본 내용도 각자 동작하기 때문에 다름
llm.invoke('내 이름은 뭔가요?').content

'미안하지만, 당신의 이름은 알 수 없습니다. 자신을 소개해 주시겠어요?'

In [5]:
messages = [
    SystemMessage(content='당신은 친절한 AI 어시스턴트입니다'),
    HumanMessage(content='내 이름은 abc입니다. 반가워요')
]

response1 = llm.invoke(messages)
print(response1.content)

안녕하세요, abc님! 반가워요. 어떻게 도와드릴까요?


In [6]:
messages.append(AIMessage(content=response1.content))
messages.append(HumanMessage(content='내 이름은 뭔가요?'))
messages

[SystemMessage(content='당신은 친절한 AI 어시스턴트입니다', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='내 이름은 abc입니다. 반가워요', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요, abc님! 반가워요. 어떻게 도와드릴까요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='내 이름은 뭔가요?', additional_kwargs={}, response_metadata={})]

In [7]:
response2 = llm.invoke(messages)
print(response2.content)

당신의 이름은 abc입니다! 어떻게 도와드릴까요?


In [12]:
import tiktoken

enc = tiktoken.encoding_for_model('gpt-4o-mini')

def count_tokens(messages):
    total = 0
    for msg in messages:
        total += len(enc.encode(msg.content))
        total += 4 # buffer
    return total

conversation = [SystemMessage(content='당신은 친절한 AI 어시스턴트입니다')]
sample_exchanges= [
    ('내 이름은 abc입니다. 반가워요', '안녕하세요, abc님! 반가워요. 어떻게 도와드릴까요?'),
    ('내 이름은 뭔가요?', '당신의 이름은 abc입니다! 어떻게 도와드릴까요?')
]

for i, (user_msg, ai_msg) in enumerate(sample_exchanges):
    conversation.append(HumanMessage(user_msg))
    conversation.append(AIMessage(ai_msg))
    tokens = count_tokens(conversation)

    print(f"{i} | {tokens} |")

0 | 53 |
1 | 84 |


In [ ]:
# Window : 대화 window 크기를 정함 -> 최신 맥락만 유지
# summarize : 10, 100 turn -> 요약 -> 새로 대화를 시작 : 디테일 유지 + 약간의 추가 토큰 비용

In [13]:
from langchain_classic.memory import ConversationBufferMemory, ConversationBufferWindowMemory

In [22]:
# 내부적으로 메모리를 가지고 있다가 사용 / summarize : 
# LangGraph : 모든 knowledge || 파이프를 하나하나의 state -> 개발자가 state에 대한 내용을 모니터링
memory = ConversationBufferMemory(return_messages = True)

In [23]:
# save_context 앞에는 Human, AI messages
memory.save_context(
    {'input' : '안녕하세요, 저는 abc입니다.'},
    {'output' : '안녕하세요, abc님! 만나서 반갑습니다.'},
)

memory.save_context(
    {'input' : '오늘 날씨가 좋네요!'},
    {'output' : '네, 정말 화창한 날씨입니다.'},
)

In [24]:
history = memory.load_memory_variables({})
history

{'history': [HumanMessage(content='안녕하세요, 저는 abc입니다.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='안녕하세요, abc님! 만나서 반갑습니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='오늘 날씨가 좋네요!', additional_kwargs={}, response_metadata={}),
  AIMessage(content='네, 정말 화창한 날씨입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [18]:
for msg in history['history']:
    print(msg.content)

안녕하세요, 저는 abc입니다.
안녕하세요, abc님! 만나서 반갑습니다.
오늘 날씨가 좋네요!
네, 정말 화창한 날씨입니다.


In [20]:
# 메모리 초기화
memory = ConversationBufferMemory(return_messages = False)
# save_context 앞에는 Human, AI messages
memory.save_context(
    {'input' : '안녕하세요, 저는 abc입니다.'},
    {'output' : '안녕하세요, abc님! 만나서 반갑습니다.'},
)

memory.save_context(
    {'input' : '오늘 날씨가 좋네요!'},
    {'output' : '네, 정말 화창한 날씨입니다.'},
)
history = memory.load_memory_variables({})
history

{'history': 'Human: 안녕하세요, 저는 abc입니다.\nAI: 안녕하세요, abc님! 만나서 반갑습니다.\nHuman: 오늘 날씨가 좋네요!\nAI: 네, 정말 화창한 날씨입니다.'}

In [25]:
from langchain_core.chat_history import InMemoryChatMessageHistory

In [26]:
chat_history = InMemoryChatMessageHistory()
chat_history.add_user_message('안녕하세요, 저는 abc입니다.')
chat_history.add_ai_message('안녕하세요, abc님! 만나서 반갑습니다.')
chat_history.add_user_message('오늘 날씨가 좋네요!')
chat_history.add_ai_message('네, 정말 화창한 날씨입니다.')

In [27]:
chat_history

InMemoryChatMessageHistory(messages=[HumanMessage(content='안녕하세요, 저는 abc입니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, abc님! 만나서 반갑습니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='오늘 날씨가 좋네요!', additional_kwargs={}, response_metadata={}), AIMessage(content='네, 정말 화창한 날씨입니다.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])

In [30]:
for msg in chat_history.messages:
    print(msg.type, msg.content)

human 안녕하세요, 저는 abc입니다.
ai 안녕하세요, abc님! 만나서 반갑습니다.
human 오늘 날씨가 좋네요!
ai 네, 정말 화창한 날씨입니다.


In [ ]:
messages = [
    chat_history.messages,
    HumanMessage(content='내 이름이 뭔가요?')
]

llm.invoke(messages)

In [ ]:
# RunnablePassthrough, RunnableBranch, RunnableLambda, RunnableParallel, RunnableWithMessageHistory

In [31]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

In [32]:
prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 친철한 AI 어시스턴트입니다.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}')
])

chain = prompt | llm

In [33]:
# history를 저장하는 곳
store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key='input',
    history_messages_key='history'
)

In [37]:
config = {'configurable' : {'session_id': 'user_001'}}
r1 = chain_with_history.invoke({'input' : '안녕하새요, 제 이름은 abc입니다.'}, config=config)

In [38]:
r1

AIMessage(content='안녕하세요, abc님! 다시 만나서 반갑습니다. 어떤 이야기를 나누고 싶으신가요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 76, 'total_tokens': 101, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_e738e3044b', 'id': 'chatcmpl-DO0DGRYrXODI14h8uyoUad93yhjbU', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d2f2e-6054-7032-8a87-af4220983055-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 76, 'output_tokens': 25, 'total_tokens': 101, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [39]:
r2 = chain_with_history.invoke({'input' : '내 이름이 뭐라고 했죠?'}, config=config)

In [40]:
r2

AIMessage(content='당신의 이름은 abc님입니다. 맞나요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 117, 'total_tokens': 129, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_162d3c480f', 'id': 'chatcmpl-DO0E2pZsefdsJT6HMlsqblsn14QDM', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d2f2f-1f13-7263-bf01-7cff58960288-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 12, 'total_tokens': 129, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [42]:
r3 = chain_with_history.invoke({'input' : '오늘 뭐하면 좋을까요?'}, config=config)

In [43]:
r3.content

'오늘의 활동으로 몇 가지를 추천해 드릴게요:\n\n1. **영화 감상**: 보고 싶었던 영화나 드라마를 선택해 보세요. 장르를 바꿔보는 것도 좋습니다!\n2. **독서**: 흥미로운 책을 읽거나 새로운 분야의 책을 탐색해 보세요.\n3. **취미 살리기**: 그림 그리기, 글쓰기, DIY 프로젝트 등 자신이 좋아하는 취미를 즐겨보세요.\n4. **산책 또는 운동**: 근처 공원이나 자연에서 산책을 하거나 집에서 운동해보세요.\n5. **요리 시도하기**: 새로운 레시피에 도전해 보거나, 좋아하는 음식을 만들어 보세요.\n6. **친구와 소통하기**: 오랜만에 연락하고 싶은 친구에게 메시지를 보내보세요.\n\n어떤 활동이 가장 마음에 드시나요?'

In [45]:
len(store['user_001'].messages)

10

In [46]:
for msg in store['user_001'].messages:
    prefix = 'customer' if msg.type == 'human' else 'sales person'
    print(f'[{prefix}] - {msg.content[:30]}')

[customer] - 안녕하새요, 제 이름은 abc입니다.
[sales person] - 안녕하세요, abc님! 만나서 반갑습니다. 어떻게 도와
[customer] - 안녕하새요, 제 이름은 abc입니다.
[sales person] - 안녕하세요, abc님! 다시 만나서 반갑습니다. 어떤 
[customer] - 내 이름이 뭐라고 했죠?
[sales person] - 당신의 이름은 abc님입니다. 맞나요?
[customer] - 오늘 뭐하면 좋을까요?
[sales person] - 오늘 할 수 있는 재미있는 활동 몇 가지를 제안해드릴게
[customer] - 오늘 뭐하면 좋을까요?
[sales person] - 오늘의 활동으로 몇 가지를 추천해 드릴게요:

1. *


In [49]:
# 윈도우를 이용해서 최근 몇개의 대화만 가져오는 내용 ConversationBufferWindowMemory
window_memory = ConversationBufferWindowMemory(k=2, return_messages=True)
window_memory.save_context({'input' : '첫번째, 내 이름은 abc입니다.'}, {'output' : '안녕하세요 abc님'})
window_memory.save_context({'input' : '두번째, 나는 학생입니다.'}, {'output' : '학생이군요! 멋지십니다'})
window_memory.save_context({'input' : '세번째, 나는 파이썬을 좋아합니다.'}, {'output' : '파이썬은 정말 재미있죠'})

In [50]:
window_memory.load_memory_variables({})

{'history': [HumanMessage(content='두번째, 나는 학생입니다.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='학생이군요! 멋지십니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='세번째, 나는 파이썬을 좋아합니다.', additional_kwargs={}, response_metadata={}),
  AIMessage(content='파이썬은 정말 재미있죠', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]}

In [51]:
from langchain_core.messages import trim_messages

messages = []

conversations = [
    ('첫번째, 내 이름은 abc입니다.', '안녕하세요 abc님'),
    ('두번째, 나는 학생입니다.', '학생이군요! 멋지십니다'),
    ('세번째, 나는 파이썬을 좋아합니다.', '파이썬은 정말 재미있죠'),
]

for user_msg, ai_msg in conversations:
    messages.append(HumanMessage(content=user_msg))
    messages.append(AIMessage(content=ai_msg))

messages

[HumanMessage(content='첫번째, 내 이름은 abc입니다.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='안녕하세요 abc님', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='두번째, 나는 학생입니다.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='학생이군요! 멋지십니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='세번째, 나는 파이썬을 좋아합니다.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='파이썬은 정말 재미있죠', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [52]:
trimmer = trim_messages(max_tokens=4, strategy='last', token_counter=len, start_on='human')
trimmed = trimmer.invoke(messages)

In [53]:
trimmed

[HumanMessage(content='두번째, 나는 학생입니다.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='학생이군요! 멋지십니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='세번째, 나는 파이썬을 좋아합니다.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='파이썬은 정말 재미있죠', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [57]:
messages = []
K = 2
def add_turn(user_input, ai_output):
    global messages
    messages.append(HumanMessage(content=user_input))
    messages.append(AIMessage(content=ai_output))

    messages = messages[-(K*2):]

add_turn('첫번째, 내 이름은 abc입니다.', '안녕하세요 abc님')
print(messages)
add_turn('두번째, 나는 학생입니다.', '학생이군요! 멋지십니다')
print(messages)
add_turn('세번째, 나는 파이썬을 좋아합니다.', '파이썬은 정말 재미있죠')
print(messages)

[HumanMessage(content='첫번째, 내 이름은 abc입니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요 abc님', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
[HumanMessage(content='첫번째, 내 이름은 abc입니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요 abc님', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='두번째, 나는 학생입니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='학생이군요! 멋지십니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
[HumanMessage(content='두번째, 나는 학생입니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='학생이군요! 멋지십니다', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='세번째, 나는 파이썬을 좋아합니다.', additional_kwargs={}, response_metadata={}), AIMessage(content='파이썬은 정말 재미있죠', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid

In [58]:
from langchain_classic.memory import ConversationSummaryMemory

In [59]:
summary_llm = ChatOpenAI(model='gpt-4o-mini')
summary_memory = ConversationSummaryMemory(
    llm = summary_llm,
    return_messages=False
)

conversations = [
    ('첫번째, 내 이름은 abc입니다.', '안녕하세요 abc님'),
    ('두번째, 나는 학생입니다.', '학생이군요! 멋지십니다'),
    ('세번째, 나는 파이썬을 좋아합니다.', '파이썬은 정말 재미있죠'),
]

for user_msg, ai_msg in conversations:
    summary_memory.save_context({'input' : user_msg}, {'output' : ai_msg})

/var/folders/wt/dlc_47_s7yv30sjq0rrctv000000gn/T/ipykernel_3163/995301399.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  summary_memory = ConversationSummaryMemory(


In [61]:
result = summary_memory.load_memory_variables({})

In [62]:
result['history']

'The human introduces themselves as "abc." The AI responds by greeting abc. The human then states that they are a student, and the AI acknowledges this and compliments them. The human expresses their enjoyment of Python, to which the AI agrees, saying that Python is really fun.'

In [63]:
class SummaryMemory:
    def __init__(self, summary_interval = 3):
        self.summary = ''
        self.recent_messages = []
        self.summary_interval = summary_interval
        self.turn_count = 0

    def _summarize(self, text):
        response = llm.invoke([
            SystemMessage(content='주어진 대화 내용을 핵심만 간결하게 요약하세요. 한국어로 작성하세요'),
            HumanMessage(content=f'기존 요약:\n{self.summary}\n\n새 대화:\n{text}')
        ])
        return response.content

    def add_exchange(self, user_msg, ai_msg):
        self.recent_messages.append(f'사용자 : {user_msg}')
        self.recent_messages.append(f'AI : {ai_msg}')
        self.turn_count += 1

        if self.turn_count % self.summary_interval == 0:
            conversation_text = '\n'.join(self.recent_messages)
            self.summary = self._summarize(conversation_text)
            self.recent_messages = []
            print(f'요약 완료 턴 {self.turn_count} 에서 요약')

    def get_context(self):
        parts = []
        if self.summary:
            parts.append(f'[이전 대화 요약] {self.summary}')
        if self.recent_messages:
            parts.append(f'[최근 대화]\n' + '\n'.join(self.recent_messages))
        return '\n\n'.join(parts)

In [64]:
smem = SummaryMemory(summary_interval=2)

In [65]:
conversations = [
    ('첫번째, 내 이름은 abc입니다.', '안녕하세요 abc님'),
    ('두번째, 나는 학생입니다.', '학생이군요! 멋지십니다'),
    ('세번째, 나는 파이썬을 좋아합니다.', '파이썬은 정말 재미있죠'),
]


for user_msg, ai_msg in conversations:
    smem.add_exchange(user_msg, ai_msg)

요약 완료 턴 2 에서 요약


In [66]:
smem.get_context()

'[이전 대화 요약] 사용자가 자신의 이름을 abc로 소개하고 학생임을 알리자 AI가 인사하며 긍정적인 반응을 보였다.\n\n[최근 대화]\n사용자 : 세번째, 나는 파이썬을 좋아합니다.\nAI : 파이썬은 정말 재미있죠'

In [69]:
class SummaryChatbot:
     def __init__(self, system_prompt = '당신은 도움이 되는 AI 어시스턴트입니다.', summary_interval = 3):
         self.system_prompt = system_prompt
         self.memory = SummaryMemory(summary_interval = summary_interval)

     def chat(self, user_input):
        context = self.memory.get_context()

        messages = [
            SystemMessage(content=self.system_prompt)
        ]

        if context:
            messages.append(SystemMessage(content=f'대화 맥락 : \n{context}'))

        messages.append(HumanMessage(content=user_input))

        response = llm.invoke(messages)
        ai_response = response.content
        self.memory.add_exchange(user_input, ai_response)
        return ai_response

In [70]:
bot = SummaryChatbot(
    system_prompt = '당신은 IT 커리어 상담사입니다.',
    summary_interval = 2
)

In [71]:
questions = [
    '안녕하세요, 백엔드 개발자 3년차인데 고민이 있습니다.',
    'AI/ML 분야로 전환을 고려 중인데 어떤 준비가 필요할까요?',
    '현재 python은 잘하는데 수학과 통계 기초가 부족합니다.',
    '온라인 강의와 학교 중 어떤 것이 효과적일까요?'
]

In [72]:
for q in questions:
    print(f'[user] {q}')
    answer = bot.chat(q)
    print(f'[상담사] {answer}')

[user] 안녕하세요, 백엔드 개발자 3년차인데 고민이 있습니다.
[상담사] 안녕하세요! 3년차 백엔드 개발자로서 어떤 고민이 있으신가요? 경력 개발, 기술 스택, 이직, 또는 다른 어떤 주제든지 말씀해 주시면 도움을 드리겠습니다.
[user] AI/ML 분야로 전환을 고려 중인데 어떤 준비가 필요할까요?
요약 완료 턴 2 에서 요약
[상담사] AI/ML 분야로의 전환은 흥미로운 도전이 될 것입니다! 다음은 준비 과정에서 고려해야 할 몇 가지 주요 단계입니다:

1. **기초 수학 및 통계 지식 강화**:
   - 머신러닝에 필요한 기본적인 수학(선형대수, 미적분학, 확률과 통계) 개념을 이해해야 합니다.

2. **프로그래밍 언어 학습**:
   - Python은 AI/ML 분야에서 가장 널리 사용되는 언어입니다. Numpy, Pandas, Matplotlib 등 데이터 분석 및 시각화를 위한 라이브러리를 익히면 좋습니다.

3. **기계학습 알고리즘 이해하기**:
   - 지도학습, 비지도학습, 강화학습 등의 기계학습 알고리즘을 배우고, 각 알고리즘의 원리와 사용 사례를 깊이 이해하는 것이 중요합니다.

4. **프레임워크 및 도구 익히기**:
   - TensorFlow, PyTorch, scikit-learn과 같은 머신러닝 프레임워크를 배우고, 실제 모델을 구현해 보는 것이 좋습니다.

5. **프로젝트 및 포트폴리오 구축**:
   - 개인 프로젝트를 통해 실제 문제를 해결하거나 Kaggle과 같은 플랫폼에서 대회에 참여하여 경험을 쌓고, 이를 포트폴리오에 추가하세요.

6. **온라인 강의 및 자료 활용**:
   - Coursera, edX, Udacity와 같은 플랫폼에서 AI/ML 관련 강의를 수강해 체계적으로 학습할 수 있습니다.

7. **커뮤니티 참여**:
   - 관련 커뮤니티나 포럼에 가입하여 다른 사람들과 경험을 공유하고, 질문을 통해 더 많이 배우는 것도 좋습니다.

8. **네트워킹**:
   - AI/ML 분야의 행사나 세미